# RAG-Augmented Cantonese Lyrics Generation with InternVL2-4B

Instead of fine-tuning, this notebook uses **Retrieval-Augmented Generation (RAG)**:
1. Build a vector index from `cantopop_corpus_final_583_yue.csv`
2. Given a query/scene description, retrieve the top-k most similar lyrics
3. Inject them as few-shot examples into the InternVL2-4B prompt

**Advantages over fine-tuning:**
- No GPU training required — runs fully at inference time
- Corpus can be updated without retraining
- Retrieved examples act as style/format anchors

> ⚠️ Run on **Google Colab with a T4/A100 GPU**. Go to `Runtime → Change runtime type → GPU`.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB" if torch.cuda.is_available() else "")

## 1. Install Dependencies

In [ ]:
!pip install -q "transformers==4.45.2" accelerate bitsandbytes sentencepiece einops torchvision \
    sentence-transformers faiss-gpu "torchao>=0.16.0"

import transformers
print("transformers:", transformers.__version__)

## 2. Clone Repo & Load Corpus

In [ ]:
import os
import pandas as pd

REPO_URL = "https://github.com/Chr1sNga1/Image2CantonSong.git"
REPO_DIR = "/content/Image2CantonSong"

if not os.path.exists(REPO_DIR):
    !git clone --branch second_model --single-branch {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

CSV_PATH = f"{REPO_DIR}/cantopop_corpus_final_583_yue.csv"
df = pd.read_csv(CSV_PATH)

# Keep only rows with meaningful lyrics
df = df.dropna(subset=["Lyrics_YuE"])
df = df[df["Lyrics_YuE"].str.strip().str.len() > 50].reset_index(drop=True)
print(f"Corpus size: {len(df)} songs")
df.head(3)

## 3. Build Vector Index

Embed each lyric using a multilingual sentence encoder, then store in a FAISS index for fast nearest-neighbour retrieval.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Multilingual model — handles Traditional Chinese well
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(EMBED_MODEL)

print("Embedding corpus...")
corpus_texts = df["Lyrics_YuE"].tolist()
embeddings = embedder.encode(corpus_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
embeddings = np.array(embeddings, dtype="float32")

# Build FAISS index (inner product = cosine similarity since embeddings are normalised)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"FAISS index built: {index.ntotal} vectors, dim={dim}")

## 4. Define Retrieval Function

In [ ]:
def retrieve(query: str, top_k: int = 3) -> list[dict]:
    """Return the top_k most similar lyrics entries for a given query."""
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = df.iloc[idx]
        results.append({
            "score": float(score),
            "lyrics": row["Lyrics_YuE"],
        })
    return results

# Quick test
test_results = retrieve("失戀的夜晚，獨自一人")
for r in test_results:
    print(f"[sim={r['score']:.3f}]")
    print(r["lyrics"][:200])
    print()

## 5. Load InternVL2-4B (4-bit)

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from transformers.modeling_utils import PreTrainedModel
from transformers.utils.quantization_config import QuantizationMethod

MODEL_ID = "OpenGVLab/InternVL2-4B"

# Patch: InternVL2's custom code calls .to() internally — forbidden on 4-bit models
_original_to = PreTrainedModel.to
def _safe_to(self, *args, **kwargs):
    if getattr(self, "quantization_method", None) == QuantizationMethod.BITS_AND_BYTES:
        return self
    return _original_to(self, *args, **kwargs)
PreTrainedModel.to = _safe_to

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModel.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
model.eval()
print("Model loaded:", MODEL_ID)
print("Device:", next(model.parameters()).device)

## 6. RAG Generation Function

For a given scene/mood description:
1. Retrieve top-k similar lyrics from the corpus
2. Build a prompt with those examples as few-shot context
3. Generate new lyrics with InternVL2-4B

In [ ]:
def build_rag_prompt(query: str, top_k: int = 3) -> str:
    """Build a RAG prompt with retrieved few-shot lyric examples."""
    examples = retrieve(query, top_k=top_k)

    few_shot = ""
    for i, ex in enumerate(examples, 1):
        # Truncate long lyrics to keep prompt manageable
        lyrics_preview = ex["lyrics"].strip()[:600]
        few_shot += f"--- 參考歌詞 {i} ---\n{lyrics_preview}\n\n"

    prompt = (
        "你是一個粵語流行歌詞創作助手。\n"
        "以下是一些優質粵語歌詞的例子，供你參考格式與風格：\n\n"
        f"{few_shot}"
        "請根據以上例子的風格，為以下主題創作一首全新的優質粵語歌詞，"
        "使用繁體中文，分為 [verse] 和 [chorus]，情感豐富、押韻自然。\n"
        f"主題：{query}\n\n"
        "請直接輸出歌詞，不要有任何額外說明。"
    )
    return prompt


def rag_generate(query: str, top_k: int = 3, max_new_tokens: int = 512) -> str:
    """Retrieve relevant lyrics and generate new lyrics with InternVL2-4B."""
    prompt = build_rag_prompt(query, top_k=top_k)

    # Use InternVL2's language model backbone for text-only generation
    lm = model.language_model

    # Format as chat template
    chat_input = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(chat_input, return_tensors="pt").to(lm.device)

    with torch.no_grad():
        output_ids = lm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print("RAG generation functions defined.")

## 7. Generate Lyrics

Enter any scene, mood, or theme description below (in Chinese or English).

In [ ]:
# ── Change this to your scene/mood description ────────────────────────────
QUERY = "失戀後漫步在雨夜的街道，回憶昔日的美好時光"
TOP_K = 3          # number of reference lyrics to retrieve
MAX_NEW_TOKENS = 512
# ──────────────────────────────────────────────────────────────────────────

print(f"Query: {QUERY}")
print(f"Retrieving top {TOP_K} reference lyrics...\n")

refs = retrieve(QUERY, top_k=TOP_K)
for i, r in enumerate(refs, 1):
    print(f"[Reference {i} | sim={r['score']:.3f}]")
    print(r["lyrics"][:300])
    print()

print("=" * 60)
print("Generating new lyrics...\n")
result = rag_generate(QUERY, top_k=TOP_K, max_new_tokens=MAX_NEW_TOKENS)
print(result)

## 8. Batch Evaluation (Optional)

Generate lyrics for multiple queries and save results to a JSON file.

In [ ]:
import json

QUERIES = [
    "失戀後漫步在雨夜的街道，回憶昔日的美好時光",
    "對故鄉的思念，在異鄉獨自打拼的心情",
    "充滿希望的夏日，年輕人追夢的故事",
]

results = []
for q in QUERIES:
    print(f"Generating for: {q}")
    lyrics = rag_generate(q, top_k=3, max_new_tokens=512)
    results.append({"query": q, "lyrics": lyrics})
    print(lyrics[:300])
    print("-" * 40)

OUTPUT_JSON = "/content/rag_generated_lyrics.json"
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"\nSaved {len(results)} results to {OUTPUT_JSON}")